In [1]:
import os
import time
import requests
import numpy as np
import pandas as pd
import yfinance as yf
import MetaTrader5 as mt5
from datetime import datetime

# ==============================================================================
# SEZIONE 1: CONFIGURAZIONE E TICKER UNIVERSE
# ==============================================================================
ticker_universe = {
    "equities_cfd": [
        "AAPL", "ADSGn", "AIRF", "ALVG", "AMZN", "BA", "BABA", "BAC", "BAYGn", "CSCO", 
        "CVX", "DBKGn", "DIS", "META", "FDX", "GE", "GM", "GOOG", "IBE", "IBM", 
        "INTC", "JNJ", "JPM", "KO", "LVMH", "MCD", "MSFT", "NFLX", "NVDA", "PFE", 
        "QCOM", "RACE", "SAN", "SIEGn", "T", "TSLA", "V", "VOWG_p", "WMT", "XOM", 
        "ZM", "RTX", "LMT", "PLTR", "AMD", "AVGO", "SBUX", "MSTR", "GME", "NKE", 
        "ARM", "SNOW", "ASML", "AZN", "BRK.B", "TTE", "BMW", "MBG"
    ],
    "crypto_cfd": [
        "BTCUSD", "DASHUSD", "ETHUSD", "LTCUSD", "XRPUSD", "XMRUSD", "NEOUSD", "ADAUSD", 
        "DOTUSD", "DOGEUSD", "SOLUSD", "AVAXUSD", "BCHUSD", "ETCUSD", "BNBUSD", "SANUSD", 
        "LNKUSD", "NERUSD", "ALGUSD", "ICPUSD", "AAVUSD", "BARUSD", "GALUSD", "GRTUSD", 
        "IMXUSD", "MANUSD", "VECUSD", "XLMUSD", "UNIUSD", "FETUSD", "XTZUSD"
    ],
    "cash_cfd": [
        "AUS200.cash", "US30.cash", "SPN35.cash", "EU50.cash", "FRA40.cash", "GER40.cash", 
        "HK50.cash", "JP225.cash", "N25.cash", "US100.cash", "US500.cash", "UK100.cash", 
        "UKOIL.cash", "USOIL.cash", "NATGAS.cash", "DXY.cash", "US2000.cash", "COCOA.c", 
        "COFFEE.c", "CORN.c", "SOYBEAN.c", "WHEAT.c", "HEATOIL.c", "COTTON.c", "SUGAR.c"
    ],
    "metals_cfd": [
        "XAGUSD", "XAGEUR", "XAGAUD", "XAUUSD", "XAUEUR", "XAUAUD", "XPDUSD", "XPTUSD", "XCUUSD"
    ],
    "exotics": [
        "EURCZK", "EURHUF", "EURNOK", "EURPLN", "USDCNH", "USDCZK", "USDHKD", "USDHUF", 
        "USDILS", "USDMXN", "USDNOK", "USDPLN", "USDSGD", "USDZAR", "USDSEK"
    ],
    "forex": [
        "AUDCAD", "AUDJPY", "AUDNZD", "AUDCHF", "AUDUSD", "GBPAUD", "GBPCAD", "GBPJPY", 
        "GBPNZD", "GBPCHF", "GBPUSD", "CADJPY", "CADCHF", "EURAUD", "EURGBP", "EURCAD", 
        "EURJPY", "EURCHF", "EURUSD", "EURNZD", "NZDCAD", "NZDCHF", "NZDJPY", "NZDUSD", 
        "CHFJPY", "USDCAD", "USDCHF", "USDJPY"
    ]
}

# Mappatura per Yahoo Finance (Azioni + Indici USA salvati da FTMO)
map_yfinance = {
    # Azioni Europee / Speciali
    "ADSGn": "ADS.DE", "AIRF": "AIR.PA", "ALVG": "ALV.DE", "BAYGn": "BAYN.DE", 
    "DBKGn": "DBK.DE", "IBE": "IBE.MC", "LVMH": "MC.PA", "SAN": "SAN.MC", 
    "SIEGn": "SIE.DE", "VOWG_p": "VOW3.DE", "ASML": "ASML.AS", "AZN": "AZN.L", 
    "TTE": "TTE.PA", "BMW": "BMW.DE", "MBG": "MBG.DE", "BRK.B": "BRK-B", "RACE": "RACE",
    # 🔥 INDICI USA SPOSTATI SU YAHOO FINANCE PER AVERE STORICO DAL 2019 🔥
    "US500.cash": "SPY",
    "US100.cash": "QQQ",
    "US30.cash": "DIA"
}

map_binance_crypto = {
    "LNKUSD": "LINKUSDT", "NERUSD": "NEARUSDT", "ALGUSD": "ALGOUSDT", 
    "ICPUSD": "ICPUSDT",   "AAVUSD": "AAVEUSDT", "MANUSD": "MANAUSDT", 
    "VECUSD": "VETUSDT",   "SANUSD": "SANDUSDT", "BARUSD": "BALUSDT",
    "FETUSD": "FETUSDT"
}

START_DATE = datetime(2019, 1, 1)
END_DATE = datetime.now()
TIMEFRAME = mt5.TIMEFRAME_D1 
BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")

# ==============================================================================
# SEZIONE 2: ARCHITETTURA EXTRACTION PROVIDERS
# ==============================================================================

def scarica_yfinance(ticker_mt5, map_dict):
    yf_ticker = map_dict.get(ticker_mt5, ticker_mt5)
    try:
        ticker_obj = yf.Ticker(yf_ticker)
        df = ticker_obj.history(start=START_DATE, end=END_DATE, auto_adjust=True)
        if not df.empty:
            df.index = df.index.tz_localize(None)
            df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
            return df
    except Exception:
        pass
    return None

def scarica_binance_public_api(ticker_mt5, map_dict):
    binance_symbol = map_dict.get(ticker_mt5, ticker_mt5.replace("USD", "USDT"))
    url = "https://api.binance.com/api/v3/klines"
    start_ts = int(START_DATE.timestamp() * 1000)
    end_ts = int(END_DATE.timestamp() * 1000)
    
    all_candles = []
    while start_ts < end_ts:
        params = {"symbol": binance_symbol, "interval": "1d", "startTime": start_ts, "limit": 1000}
        try:
            res = requests.get(url, params=params)
            if res.status_code != 200: break
            data = res.json()
            if not data or len(data) == 0: break
            all_candles.extend(data)
            start_ts = data[-1][0] + 86400000 
            time.sleep(0.05)
        except Exception:
            break
            
    if len(all_candles) == 0: return None
        
    columns = ['time', 'Open', 'High', 'Low', 'Close', 'Volume', 'close_time', 'q_volume', 'trades', 'taker_base', 'taker_quote', 'ignore']
    df = pd.DataFrame(all_candles, columns=columns)
    df['time'] = pd.to_datetime(df['time'], unit='ms')
    df.set_index('time', inplace=True)
    
    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        df[col] = df[col].astype(float)
    return df[['Open', 'High', 'Low', 'Close', 'Volume']]

# ==============================================================================
# SEZIONE 3: CORE DATA INGESTION PIPELINE (CON FORZATURA FLUSH SE CAMBIA ROUTING)
# ==============================================================================

def initialize_and_download_v3(universe):
    all_data = {cluster: {} for cluster in universe.keys()}
    
    # Controlliamo se gli indici sono già stati convertiti in locale, altrimenti forziamo scaricamento
    path_check_spy = os.path.join(BASE_DIR, "cash_cfd", "US500.cash.parquet")
    is_cached = all(os.path.exists(os.path.join(BASE_DIR, c)) for c in universe.keys())
    
    if is_cached and os.path.exists(path_check_spy):
        # Se il file di SPY esiste, controlliamo se ha lo storico lungo (segno che è il dato Yahoo)
        df_check = pd.read_parquet(path_check_spy)
        if len(df_check) > 1500: 
            total_loaded = 0
            for cluster in universe.keys():
                cluster_path = os.path.join(BASE_DIR, cluster)
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        ticker = file.replace(".parquet", "")
                        all_data[cluster][ticker] = pd.read_parquet(os.path.join(cluster_path, file))
                        total_loaded += 1
            print(f"📦 [CACHE SINCRONIZZATO] Caricati {total_loaded} file Parquet (Indici USA inclusi dal 2019).")
            return all_data

    print("⏳ Aggiornamento archivio locale in corso per allineamento Indici 2019...")
    total_tickers = sum(len(tickers) for tickers in universe.values())
    counter = 0

    os.makedirs(BASE_DIR, exist_ok=True)
    for cluster in universe.keys(): os.makedirs(os.path.join(BASE_DIR, cluster), exist_ok=True)

    print("🔌 Connessione a MT5 FTMO in corso...")
    mt5_active = mt5.initialize()

    for cluster, tickers in universe.items():
        print(f"\n{'='*60}\nCLUSTER: {cluster.upper()}\n{'='*60}")
        theoretical_freq = 'D' if cluster == "crypto_cfd" else 'B'
        
        for ticker in tickers:
            counter += 1
            df = None
            provider = ""
            
            # ROUTING MODIFICATO: Se il ticker è un indice USA target, va su Yahoo Finance!
            if cluster == "equities_cfd" or ticker in ["US500.cash", "US100.cash", "US30.cash"]:
                df = scarica_yfinance(ticker, map_yfinance)
                provider = "Yahoo Finance (Deep History)"
                
            elif cluster == "crypto_cfd":
                df = scarica_binance_public_api(ticker, map_binance_crypto)
                provider = "Binance API (24/7)"
                
            elif mt5_active: 
                if mt5.symbol_select(ticker, True):
                    rates = mt5.copy_rates_range(ticker, TIMEFRAME, START_DATE, END_DATE)
                    if rates is not None and len(rates) > 0:
                        df = pd.DataFrame(rates)
                        df['time'] = pd.to_datetime(df['time'], unit='s')
                        df.set_index('time', inplace=True)
                        df = df[['open', 'high', 'low', 'close', 'tick_volume']]
                        df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
                        provider = "MT5 FTMO"
            
            if df is not None and len(df) > 0:
                all_data[cluster][ticker] = df
                df.to_parquet(os.path.join(BASE_DIR, cluster, f"{ticker}.parquet"))
                print(f"[{counter}/{total_tickers}] ✅ {ticker:<11} | Prov: {provider[:15]}... | Oss: {len(df)}")
            else:
                print(f"[{counter}/{total_tickers}] ❌ {ticker:<11} | NON TROVATO.")
                
    if mt5_active: mt5.shutdown()
    return all_data

dati_ram = initialize_and_download_v3(ticker_universe)

⏳ Aggiornamento archivio locale in corso per allineamento Indici 2019...
🔌 Connessione a MT5 FTMO in corso...

CLUSTER: EQUITIES_CFD
[1/166] ✅ AAPL        | Prov: Yahoo Finance (... | Oss: 1872
[2/166] ✅ ADSGn       | Prov: Yahoo Finance (... | Oss: 1892
[3/166] ✅ AIRF        | Prov: Yahoo Finance (... | Oss: 1907
[4/166] ✅ ALVG        | Prov: Yahoo Finance (... | Oss: 1892
[5/166] ✅ AMZN        | Prov: Yahoo Finance (... | Oss: 1872
[6/166] ✅ BA          | Prov: Yahoo Finance (... | Oss: 1872
[7/166] ✅ BABA        | Prov: Yahoo Finance (... | Oss: 1872
[8/166] ✅ BAC         | Prov: Yahoo Finance (... | Oss: 1872
[9/166] ✅ BAYGn       | Prov: Yahoo Finance (... | Oss: 1892
[10/166] ✅ CSCO        | Prov: Yahoo Finance (... | Oss: 1872
[11/166] ✅ CVX         | Prov: Yahoo Finance (... | Oss: 1872
[12/166] ✅ DBKGn       | Prov: Yahoo Finance (... | Oss: 1892
[13/166] ✅ DIS         | Prov: Yahoo Finance (... | Oss: 1872
[14/166] ✅ META        | Prov: Yahoo Finance (... | Oss: 1872
[15/166]

In [2]:
# ==============================================================================
# CELLA 2: DATA CLEANING & ALIGNMENT ENGINE (DINAMICO ASINCRONO)
# ==============================================================================
import pandas as pd
import numpy as np

# Inizializziamo la struttura pulita partendo da dati_ram
dati_puliti = {cluster: {} for cluster in dati_ram.keys()}

# Soglie calibrate per includere gli Indici USA storici, il GER40 e le commodity corte
MIN_BARS = 300             # Richiesti circa 1 anno e due mesi di dati minimi
MAX_MISSING_ALLOWED = 20.0 # Tolleranza alzata al 20% (Salva il GER40 che ha il 16.68% di buchi)

print("🧹 Avvio Engine di Pulizia Dinamica (Filtro Asincrono)...")
print(f"Filtro Anzianità: Gli asset devono possedere ALMENO {MIN_BARS} osservazioni.")
print(f"Filtro Tolleranza: Massimo {MAX_MISSING_ALLOWED}% di dati mancanti.\n")

stats_pulizia = {cluster: {"accettati": 0, "scartati_anzianita": 0, "scartati_buchi": 0} for cluster in dati_ram.keys()}
scartati_dettaglio = {cluster: {"anzianita": [], "buchi": []} for cluster in dati_ram.keys()}

for cluster, tickers in dati_ram.items():
    theoretical_freq = 'D' if cluster == "crypto_cfd" else 'B'
    
    for ticker, df_raw in tickers.items():
        df = df_raw.copy()
        
        # 1. Filtro Anzianità sulle osservazioni totali disponibili
        if len(df) < MIN_BARS:
            stats_pulizia[cluster]["scartati_anzianita"] += 1
            scartati_dettaglio[cluster]["anzianita"].append(f"{ticker} (Oss: {len(df)})")
            continue
            
        # 2. Calcolo dei dati mancanti sul calendario naturale dell'asset
        n_obs = len(df)
        ideal_calendar = pd.date_range(start=df.index.min(), end=df.index.max(), freq=theoretical_freq)
        missing_days = len(ideal_calendar) - n_obs
        pct_missing = max(0.0, (missing_days / len(ideal_calendar)) * 100 if len(ideal_calendar) > 0 else 0)
        
        # 3. Controllo Soglia Buchi
        if pct_missing > MAX_MISSING_ALLOWED:
            stats_pulizia[cluster]["scartati_buchi"] += 1
            scartati_dettaglio[cluster]["buchi"].append(f"{ticker} (Buchi: {pct_missing:.2f}%)")
            continue
            
        # 4. Data Repair standard e pulizia strutturale
        df = df.dropna(subset=['Close'])
        df = df.sort_index()
        
        dati_puliti[cluster][ticker] = df
        stats_pulizia[cluster]["accettati"] += 1

# --- REPORT VISIVO DEGLI ASSET ACCETTATI E SCARTATI ---
print("="*80)
print(f"📊 REPORT FINALE DI FILTRAGGIO DATASET SEZIONE 3")
print("="*80)
for cluster, stats in stats_pulizia.items():
    print(f"📁 Cluster: {cluster.upper()} | Accettati con successo: {stats['accettati']}")
    if scartati_dettaglio[cluster]["anzianita"]:
        print(f"   ⚠️ Scartati Storico Corto: {', '.join(scartati_dettaglio[cluster]['anzianita'])}")
    if scartati_dettaglio[cluster]["buchi"]:
        print(f"   ❌ Scartati Troppi Buchi: {', '.join(scartati_dettaglio[cluster]['buchi'])}")
    print("-"*80)

🧹 Avvio Engine di Pulizia Dinamica (Filtro Asincrono)...
Filtro Anzianità: Gli asset devono possedere ALMENO 300 osservazioni.
Filtro Tolleranza: Massimo 20.0% di dati mancanti.

📊 REPORT FINALE DI FILTRAGGIO DATASET SEZIONE 3
📁 Cluster: EQUITIES_CFD | Accettati con successo: 58
--------------------------------------------------------------------------------
📁 Cluster: CRYPTO_CFD | Accettati con successo: 31
--------------------------------------------------------------------------------
📁 Cluster: CASH_CFD | Accettati con successo: 25
--------------------------------------------------------------------------------
📁 Cluster: METALS_CFD | Accettati con successo: 9
--------------------------------------------------------------------------------
📁 Cluster: EXOTICS | Accettati con successo: 15
--------------------------------------------------------------------------------
📁 Cluster: FOREX | Accettati con successo: 28
-----------------------------------------------------------------------

In [5]:
import os
import numpy as np
import pandas as pd

# ==============================================================================
# PROTEZIONE DA NAMEERROR: AUTO-RECOVER DA DISCO SE RAM VUOTA
# ==============================================================================
BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")

if 'dati_puliti' not in locals() and 'dati_puliti' not in globals():
    print("📦 'dati_puliti' non rilevata in RAM. Inizio lettura dinamica dai singoli file Parquet locali...")
    dati_puliti = {}
    if os.path.exists(BASE_DIR):
        for cluster in os.listdir(BASE_DIR):
            cluster_path = os.path.join(BASE_DIR, cluster)
            if os.path.isdir(cluster_path):
                dati_puliti[cluster] = {}
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        ticker = file.replace(".parquet", "")
                        dati_puliti[cluster][ticker] = pd.read_parquet(os.path.join(cluster_path, file))
        total_letti = sum(len(tickers) for tickers in dati_puliti.values())
        print(f"✅ Ripristinati con successo {total_letti} asset da disco. Procedo con l'analisi quantitativa.")
    else:
        print(f"❌ ERRORE CRITICO: Cartella {BASE_DIR} non trovata. Esegui prima il download dei dati.")

# ==============================================================================
# FUNZIONI QUANTITATIVE CORE (VERSIONE PRO)
# ==============================================================================

def calcola_hurst_veloce(price_series):
    """
    Calcola l'esponente di Hurst usando il Variance-Time Plot.
    H < 0.5: Mean Reverting | H = 0.5: Random Walk | H > 0.5: Trending
    """
    log_prices = np.log(price_series).dropna()
    if len(log_prices) < 200:
        return np.nan
        
    lags = [2, 4, 8, 16, 32, 64, 128]
    stds = []
    
    for lag in lags:
        diff = log_prices.diff(lag).dropna()
        std_val = diff.std()
        if std_val > 0:
            stds.append(std_val)
        else:
            stds.append(1e-8)
        
    poly = np.polyfit(np.log(lags), np.log(stds), 1)
    return poly[0]

def calcola_kaufman_er(df, window=20):
    """
    Calcola l'Efficiency Ratio di Kaufman mediano.
    Vicino a 1: Trend pulito | Vicino a 0: Puro rumore caotico
    """
    close = df['Close']
    movimento_netto = close.diff(window).abs()
    rumore_totale = close.diff().abs().rolling(window).sum()
    
    er = movimento_netto / rumore_totale.replace(0, np.nan)
    return er.median()

def calcola_volatilità_annuale(df, is_crypto=False):
    """Calcola la volatilità annualizzata classica"""
    log_returns = np.log(df['Close'] / df['Close'].shift(1)).dropna()
    giorni_anno = 365 if is_crypto else 252
    return log_returns.std() * np.sqrt(giorni_anno)

# ==============================================================================
# ENGINE DI CALCOLO ADATTATO PER DATI LOCALI/RAM
# ==============================================================================

analisi_risultati = []

for cluster, tickers in dati_puliti.items():
    is_crypto = (cluster == "crypto_cfd")
    
    for ticker, df in tickers.items():
        if df is None or df.empty:
            continue
            
        # --- ADATTAMENTO DI SICUREZZA ---
        # 1. Copia profonda per evitare di sporcare il dataset originale
        df_working = df.copy()
        # 2. Forziamo l'indice in formato Datetime (vitale se ricaricato da Parquet)
        df_working.index = pd.to_datetime(df_working.index)
        # 3. ORDINAMENTO CRONOLOGICO (Evita che .diff() calcoli linee temporali sballate)
        df_working = df_working.sort_index()
        
        # Esecuzione dei calcoli statistici
        if 'Close' in df_working.columns and len(df_working) >= 200:
            h_exp = calcola_hurst_veloce(df_working['Close'])
            k_er = calcola_kaufman_er(df_working, window=20)
            vol_ann = calcola_volatilità_annuale(df_working, is_crypto=is_crypto)
            
            analisi_risultati.append({
                "Ticker": ticker,
                "Cluster": cluster,
                "Hurst": round(h_exp, 3) if not np.isnan(h_exp) else None,
                "Kaufman_ER": round(k_er, 3),
                "Vol_Annualizzata_%": round(vol_ann * 100, 2),
                "Inizio_Storico": df_working.index.min().strftime('%Y-%m-%d'),
                "Fine_Storico": df_working.index.max().strftime('%Y-%m-%d')
            })

# Creazione della matrice finale ordinata per Hurst decrescente
if analisi_risultati:
    df_dna = pd.DataFrame(analisi_risultati).sort_values(by="Hurst", ascending=False).reset_index(drop=True)
    print(f"🧬 Mappatura genetica completata con successo su {len(df_dna)} asset validati.")
else:
    print("⚠️ Attenzione: Nessun dato valido elaborato dai file Parquet.")

🧬 Mappatura genetica completata con successo su 166 asset validati.


In [6]:
# ==============================================================================
# SEZIONE 4: PRINT ENGINE - RISULTATI STRUTTURATI PER CLUSTER
# ==============================================================================

# Estraiamo i cluster unici presenti nel dataset
cluster_unici = df_dna['Cluster'].unique()

print("📊 ANALISI STRUTTURALE DEL PORTAFOGLIO (Sorted by Hurst Exponent)\n")

for cluster in cluster_unici:
    print("\n" + "="*75)
    print(f"📂 CLUSTER: {cluster.upper()}")
    print("="*75)
    
    # Filtriamo la matrice globale per il cluster corrente 
    # Ordiniamo dal valore di Hurst più alto (Trending) a quello più basso (Mean Reverting)
    df_filtrato = df_dna[df_dna['Cluster'] == cluster].sort_values(by="Hurst", ascending=False)
    
    # Rinominiamo temporaneamente le colonne per un output della tabella ultra-pulito
    tabella_stampa = df_filtrato[[
        "Ticker", "Hurst", "Kaufman_ER", "Vol_Annualizzata_%", "Inizio_Storico"
    ]].copy()
    
    tabella_stampa.columns = ["TICKER", "HURST", "KAUFMAN ER", "VOL ANN %", "START DATE"]
    
    # STAMPA FORMATTATA: Forziamo i decimali fissi per evitare disallineamenti estetici
    print(tabella_stampa.to_string(
        index=False, 
        justify='left', 
        col_space=12,
        formatters={
            'HURST': lambda x: f"{x:.3f}" if pd.notnull(x) else "NaN",
            'KAUFMAN ER': lambda x: f"{x:.3f}" if pd.notnull(x) else "NaN",
            'VOL ANN %': lambda x: f"{x:.2f}%" if pd.notnull(x) else "NaN"
        }
    ))
    
    # --- METRICHE DI SINTESI DEL CLUSTER ---
    # Usiamo .mean() di pandas che esclude automaticamente i valori nulli/NaN senza andare in errore
    hurst_medio = df_filtrato['Hurst'].mean()
    er_medio = df_filtrato['Kaufman_ER'].mean()
    vol_media = df_filtrato['Vol_Annualizzata_%'].mean()
    
    # Interpretazione rapida del regime del cluster
    if pd.isna(hurst_medio):
        regime_str = "INSUFFICIENTE (Dati non validi)"
    elif hurst_medio > 0.52:
        regime_str = "TRENDING (Persistente)"
    elif hurst_medio < 0.48:
        regime_str = "MEAN REVERTING (Anti-persistente)"
    else:
        regime_str = "RANDOM WALK (Rumore/Equilibrio)"
        
    print("-" * 75)
    print(f"📈 SINTESI MACRO [{cluster.upper()}]:")
    print(f"   • Regime Prevalente:  {regime_str}")
    print(f"   • Hurst Medio:        {hurst_medio:.3f}" if pd.notnull(hurst_medio) else "   • Hurst Medio:        NaN")
    print(f"   • Kaufman ER Medio:   {er_medio:.3f}" if pd.notnull(er_medio) else "   • Kaufman ER Medio:   NaN")
    print(f"   • Volatilità Media:   {vol_media:.2f}%" if pd.notnull(vol_media) else "   • Volatilità Media:   NaN")
    print("="*75 + "\n")

📊 ANALISI STRUTTURALE DEL PORTAFOGLIO (Sorted by Hurst Exponent)


📂 CLUSTER: CRYPTO_CFD
TICKER       HURST        KAUFMAN ER   VOL ANN %    START DATE  
 SOLUSD      0.586        0.225        115.81%      2020-08-11  
 BNBUSD      0.576        0.211         83.65%      2019-01-01  
AVAXUSD      0.575        0.216        114.77%      2020-09-22  
 ADAUSD      0.563        0.212         98.02%      2019-01-01  
 FETUSD      0.562        0.209        137.93%      2019-02-28  
 SANUSD      0.562        0.216        129.16%      2020-08-14  
DOGEUSD      0.553        0.221        127.54%      2019-07-05  
 BTCUSD      0.550        0.211         62.94%      2019-01-01  
 MANUSD      0.544        0.198        120.61%      2020-08-06  
 VECUSD      0.541        0.214        107.44%      2019-01-01  
 DOTUSD      0.538        0.215        101.51%      2020-08-18  
 UNIUSD      0.527        0.200        114.98%      2020-09-17  
 NERUSD      0.525        0.235        121.77%      2020-10-14  
 

In [9]:
# ==============================================================================
# SEZIONE 6: ALGORITHMIC TREND MAPPING (3 TIERS - AGGIORNATO CON AUTOMATIC OVERRIDE)
# ==============================================================================
import numpy as np
import pandas as pd

SOGLIA_HURST = 0.50
SOGLIA_KAUFMAN = 0.21

def categorizza_trend_v3(row):
    h = row['Hurst']
    er = row['Kaufman_ER']
    cluster = row['Cluster']
    
    # 1. TIER 1: Trend Persistente e Pulito (Aperto a tutti gli asset)
    if h is not None and not pd.isna(h) and h >= SOGLIA_HURST and er >= SOGLIA_KAUFMAN:
        return "Tier 1: Strong Trend (Direzionale Puro)"
    
    # 🚨 OVERRIDE STRATEGICO AUTOMATICO PER LE EQUITIES 🚨
    # Se è un'azione e non ha superato il Tier 1, viene forzata d'ufficio nel Tier 2
    # Impedendo qualsiasi classificazione in Tier 3 a prescindere dalle metriche numeriche
    if cluster == "equities_cfd":
        return "Tier 2: Mid Trend (Grinding / Noisy Trend)"
    
    # --------------------------------------------------------------------------
    # REGOLE STANDARD (Valide solo per Crypto, Forex, Commodities, Indici Cash)
    # --------------------------------------------------------------------------
    if h is None or pd.isna(h) or h == 0:
        return "Tier 3: No Trend (Escluso / Rumore)"
        
    # TIER 2 STANDARD: Grinding Trend o Noisy Trend
    if (h < SOGLIA_HURST and er >= SOGLIA_KAUFMAN) or (h >= SOGLIA_HURST and er < SOGLIA_KAUFMAN):
        return "Tier 2: Mid Trend (Grinding / Noisy Trend)"
    
    # TIER 3 STANDARD: Pura Mean Reversion o Lateralizzazione caotica
    return "Tier 3: No Trend (Mean Reversion Strutturale)"

# Applichiamo la nuova logica deterministica
df_dna['Categoria_Trend'] = df_dna.apply(categorizza_trend_v3, axis=1)

# --- EMISSIONE DEL REPORT AGGIORNATO ---
print("🎯 NUOVA MAPPATURA OPERATIVA (Tier 2 Esteso con Override Equities)\n")

categorie_ordinate = [
    "Tier 1: Strong Trend (Direzionale Puro)",
    "Tier 2: Mid Trend (Grinding / Noisy Trend)",
    "Tier 3: No Trend (Mean Reversion Strutturale)"
]

for cat in categorie_ordinate:
    df_cat = df_dna[df_dna['Categoria_Trend'] == cat]
    total_asset = len(df_cat)
    
    print("=" * 80)
    print(f"🔥 {cat.upper()} | Totale Asset: {total_asset}")
    print("=" * 80)
    
    if total_asset > 0:
        for cluster, group in df_cat.groupby('Cluster'):
            ticker_list = group['Ticker'].tolist()
            print(f"  📁 [{cluster.upper()}]:")
            print(f"     {', '.join(ticker_list)}")
    else:
        print("  Nessun asset in questa categoria.")
    print("\n")

🎯 NUOVA MAPPATURA OPERATIVA (Tier 2 Esteso con Override Equities)

🔥 TIER 1: STRONG TREND (DIREZIONALE PURO) | Totale Asset: 23
  📁 [CASH_CFD]:
     COCOA.c
  📁 [CRYPTO_CFD]:
     SOLUSD, BNBUSD, AVAXUSD, ADAUSD, SANUSD, DOGEUSD, BTCUSD, VECUSD, DOTUSD, NERUSD, ETHUSD, IMXUSD, XLMUSD
  📁 [EQUITIES_CFD]:
     ZM, NFLX, MSTR, TSLA, META, FDX, BAYGn, PLTR, NVDA


🔥 TIER 2: MID TREND (GRINDING / NOISY TREND) | Totale Asset: 67
  📁 [CASH_CFD]:
     COFFEE.c, SOYBEAN.c, US100.cash, US500.cash, US30.cash
  📁 [CRYPTO_CFD]:
     FETUSD, MANUSD, UNIUSD, ALGUSD, ETCUSD, LNKUSD, BARUSD, NEOUSD, GRTUSD, GALUSD
  📁 [EQUITIES_CFD]:
     GE, SAN, XOM, AIRF, DIS, INTC, GM, LVMH, MBG, GOOG, AMD, JPM, ASML, ADSGn, GME, T, BAC, BA, NKE, AMZN, RTX, SBUX, SIEGn, CSCO, MSFT, AAPL, BMW, BRK.B, LMT, BABA, RACE, QCOM, CVX, PFE, TTE, SNOW, VOWG_p, ARM, JNJ, ALVG, DBKGn, AZN, IBE, KO, AVGO, WMT, IBM, MCD, V
  📁 [EXOTICS]:
     USDHUF
  📁 [FOREX]:
     AUDNZD
  📁 [METALS_CFD]:
     XCUUSD


🔥 TIER 3: NO TREND (MEA

In [10]:
import json
import os

# 1. Generazione del dizionario con liste piatte di ticker pronti per il loop di trading
mappa_strategie = {
    "tier_1_strong_trend": df_dna[df_dna['Categoria_Trend'].str.contains("Tier 1")]['Ticker'].tolist(),
    "tier_2_mid_trend":    df_dna[df_dna['Categoria_Trend'].str.contains("Tier 2")]['Ticker'].tolist(),
    "tier_3_no_trend":     df_dna[df_dna['Categoria_Trend'].str.contains("Tier 3")]['Ticker'].tolist()
}

# 2. Salvataggio in formato JSON nella cartella del progetto
cartella_data = "progetto_trading_data"
os.makedirs(cartella_data, exist_ok=True)
file_json = os.path.join(cartella_data, "mappa_strategie.json")

with open(file_json, "w") as f:
    json.dump(mappa_strategie, f, indent=4)

print("📌 DIZIONARIO STRATEGIE GENERATO IN RAM:")
print(f"   • Ticker in Tier 1 (Strong): {len(mappa_strategie['tier_1_strong_trend'])}")
print(f"   • Ticker in Tier 2 (Mid):    {len(mappa_strategie['tier_2_mid_trend'])}")
print(f"   • Ticker in Tier 3 (No):     {len(mappa_strategie['tier_3_no_trend'])}")
print(f"\n💾 File di configurazione salvato localmente: {file_json}")

📌 DIZIONARIO STRATEGIE GENERATO IN RAM:
   • Ticker in Tier 1 (Strong): 23
   • Ticker in Tier 2 (Mid):    67
   • Ticker in Tier 3 (No):     76

💾 File di configurazione salvato localmente: progetto_trading_data\mappa_strategie.json
